### Ingestion del archivo "movie_genre.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
movie_genre_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("genreId", IntegerType(), True)
])

In [0]:
movie_genre_df = spark.read \
            .schema(movie_genre_schema) \
            .json(f"{bronze_folder_path}/{v_file_date}/movie_genre.json")

In [0]:
movie_genre_df.printSchema()

In [0]:
display(movie_genre_df)

#### Paso 2 - renombrar columnas y agregar columnas nuevas
1. renombrar "movieId" y "genreId"
2. Agregar las columnas "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
mov_gen_with_columns_df = add_ingestion_date(movie_genre_df) \
                            .withColumnsRenamed({"movieId": "movie_id"}) \
                            .withColumnsRenamed({"genreId": "genre_id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(mov_gen_with_columns_df)

#### Paso 4 - Escribir salida en un formato "Parquet"

In [0]:
# mov_gen_with_columns_df.write.mode("overwrite").partitionBy("movie_id").parquet(f"{silver_folder_path}/movies_genres")

In [0]:
# %fs
# ls "abfss://silver@moviehistory22.dfs.core.windows.net/movies_genres"

In [0]:
# display(mov_gen_with_columns_df)

In [0]:
# overwrite_partition("movie_silver", "movies_genres","file_date",v_file_date)

In [0]:
# mov_gen_with_columns_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_genres")
# mov_gen_with_columns_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_genres")

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id AND tgt.file_date=src.file_date'
merge_delta_lake(mov_gen_with_columns_df, "movie_silver", "movies_genres", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies_genres
GROUP BY file_date

In [0]:
%sql
SELECT * FROM movie_silver.movies_genres

In [0]:
dbutils.notebook.exit("Success")